### Kernel: Python

## Treinamento da rede e avaliacao para os dados de teste

In [31]:
# Configurações do ambiente em Python

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 2003
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.0025

np.random.seed(SEED)
torch.manual_seed(SEED)

In [32]:
# Carregamento dos dados

entrada_rede = pd.read_csv("../dados/entrada/entrada_rede.csv")

# Split treino/teste
dados_treino = entrada_rede.loc[entrada_rede["conjunto"] == "treino"].copy()
dados_teste = entrada_rede.loc[entrada_rede["conjunto"] == "teste"].copy()

In [33]:
dados_treino.head(20)

,id_paciente,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,...,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,tempo,pseudo,conjunto
0,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,2.0,1.000004,treino
1,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,7.0,1.000116,treino
2,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,11.0,1.000329,treino
3,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,15.0,1.000705,treino
4,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,20.0,-0.038298,treino
5,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,27.0,-0.037759,treino
6,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,32.1,-0.037240,treino
7,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,41.0,-0.036584,treino
8,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,52.9,-0.035918,treino
9,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,3,Média Renda,5,2014,8500,12,42.0,73.0,-0.034966,treino


In [34]:
dados_teste.head(20)

,id_paciente,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,...,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,tempo,pseudo,conjunto
48610,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,2.0,NaN,teste
48611,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,7.0,NaN,teste
48612,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,11.0,NaN,teste
48613,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,15.0,NaN,teste
48614,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,20.0,NaN,teste
48615,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,27.0,NaN,teste
48616,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,32.1,NaN,teste
48617,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,41.0,NaN,teste
48618,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,52.9,NaN,teste
48619,3,Não Branca,0,2,Não,Nunca,Nunca,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0.0,73.0,NaN,teste


In [35]:
# Sequência de tempos para avaliação da rede nos dados de teste

tempos_avaliacao_teste = np.arange(1, 132, 1)
tempos_avaliacao_teste

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131])

## Treinamento e Previsões para os dados de teste das Redes Neurais

In [36]:
#Definição da estrutura da rede neural

class rede_sobrevivencia(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 2 * input_dim),
            nn.Tanh(),
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## Rede Neural A: one-hot encoding

In [37]:
def normaliza_coluna_numerica(serie, stats_coluna):
    valores = pd.to_numeric(serie, errors="coerce").fillna(stats_coluna["mediana"])
    return (valores - stats_coluna["media"]) / (stats_coluna["desvio"] + 1e-8)


covariaveis = [col for col in dados_treino.columns if col not in ["id_paciente", "tempo", "pseudo", "conjunto"]]
covariaveis_numericas = [col for col in covariaveis if pd.api.types.is_numeric_dtype(dados_treino[col])]
covariaveis_categoricas = [col for col in covariaveis if col not in covariaveis_numericas]

estatisticas_numericas = {}
for col in covariaveis_numericas:
    serie = pd.to_numeric(dados_treino[col], errors="coerce")
    mediana = float(serie.median()) if serie.notna().any() else 0.0
    serie_imp = serie.fillna(mediana)
    estatisticas_numericas[col] = {
        "mediana": mediana,
        "media": float(serie_imp.mean()),
        "desvio": float(serie_imp.std(ddof=0)),
    }

mapeamento_categorias = {}
for col in covariaveis_categoricas:
    mapeamento_categorias[col] = sorted(dados_treino[col].fillna("NA").astype(str).unique().tolist())


def gera_matriz_design_one_hot(dados, tempos):
    dados_ref = dados.reset_index(drop=True).copy()

    base_num = pd.DataFrame(index=dados_ref.index)
    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(dados_ref[col], estatisticas_numericas[col])

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()
        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )
        base_cat = pd.get_dummies(base_cat, prefix=covariaveis_categoricas, dtype=float)
        base_cat = base_cat.reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    tcat = pd.Categorical(
        pd.to_numeric(dados_ref["tempo"], errors="coerce").astype(float),
        categories=tempos,
        ordered=True,
    )
    t_oh = pd.get_dummies(tcat, prefix="tempo", dtype=float).reset_index(drop=True)

    return pd.concat([base_num, base_cat, t_oh], axis=1)


time_levels = np.sort(dados_treino["tempo"].astype(float).unique())
x_treino_onehot = gera_matriz_design_one_hot(dados_treino, time_levels)
y_treino_onehot = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_onehot = torch.tensor(x_treino_onehot.values.astype(np.float32))
y_train_onehot = torch.tensor(y_treino_onehot.astype(np.float32))

train_loader_onehot = DataLoader(
    TensorDataset(X_train_onehot, y_train_onehot),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

print(f"dimensao X_train: {tuple(X_train_onehot.shape)}")
print(f"dimensao y_train: {tuple(y_train_onehot.shape)}")

dimensao X_train: (48610, 55)
dimensao y_train: (48610, 1)


In [38]:
# One hot
x_treino_onehot.head(20)

,instrucao,est_cong,rhc_idade_no_diagnostico_tumor,macroregiao,faixa_etcat,ano,diff_data_1consulta,diff_data_tratamento,rhc_raca_cor_Branca,rhc_raca_cor_Não Branca,...,tempo_2.0,tempo_7.0,tempo_11.0,tempo_15.0,tempo_20.0,tempo_27.0,tempo_32.1,tempo_41.0,tempo_52.9,tempo_73.0
0,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,-1.504404,0.889058,1.230294,1.718244,1.501794,-0.258050,-0.226377,-0.458714,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [39]:
# Definicao do modelo, criterio e otimizador para a rede one-hot
model_onehot = rede_sobrevivencia(input_dim=X_train_onehot.shape[1])
criterion_onehot = nn.MSELoss()
optimizer_onehot = optim.Adam(model_onehot.parameters(), lr=LEARNING_RATE)

In [40]:
# Treinamento da rede

for epoch in range(EPOCHS):
    model_onehot.train()
    running_loss = 0.0

    for xb, yb in train_loader_onehot:
        optimizer_onehot.zero_grad()
        pred = model_onehot(xb)
        loss = criterion_onehot(pred, yb)
        loss.backward()
        optimizer_onehot.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Época {epoch + 1:4d}/{EPOCHS} | mse={(running_loss / len(train_loader_onehot)):.6f}")

Época   10/100 | mse=0.040283
Época   20/100 | mse=0.023332
Época   30/100 | mse=0.018830
Época   40/100 | mse=0.016962
Época   50/100 | mse=0.015828
Época   60/100 | mse=0.014742
Época   70/100 | mse=0.014439
Época   80/100 | mse=0.013871
Época   90/100 | mse=0.014042
Época  100/100 | mse=0.013709


In [41]:
# Pasta de saída das predições
dir_saida = Path("../dados/saida")
dir_saida.mkdir(parents=True, exist_ok=True)

# IDs únicos de teste
dados_ids_teste = dados_teste[["id_paciente"] + covariaveis].drop_duplicates().sort_values("id_paciente")

# Base com os tempos de avaliacao do teste
linhas_teste_evento = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_evento in tempos_avaliacao_teste:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_evento),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_teste_evento.append(linha)
dados_teste_evento = pd.DataFrame(linhas_teste_evento)

# Previsões Rede one-hot
linhas_grade_A = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_grade in time_levels:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_grade),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_grade_A.append(linha)

grade_predicao_A = pd.DataFrame(linhas_grade_A)
x_pred_A = gera_matriz_design_one_hot(grade_predicao_A, time_levels)
x_pred_A = x_pred_A.reindex(columns=x_treino_onehot.columns, fill_value=0.0)

# Predição da Rede one-hot na grade de treino
model_onehot.eval()
with torch.no_grad():
    grade_predicao_A["pred_s"] = model_onehot(torch.tensor(x_pred_A.values.astype(np.float32))).cpu().numpy().reshape(-1)

grade_predicao_A = grade_predicao_A.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

# Predição da Rede one-hot nos tempos de avaliacao do teste (via interpolação)
linhas_evento_A = []
for id_individuo, predicoes_individuo in grade_predicao_A.groupby("id_paciente"):
    predicoes_individuo = predicoes_individuo.sort_values("tempo")
    predicoes_evento = np.interp(
        tempos_avaliacao_teste,
        predicoes_individuo["tempo"].values,
        predicoes_individuo["pred_s"].values,
    )
    for tempo_evento, valor_predito in zip(tempos_avaliacao_teste, predicoes_evento):
        linhas_evento_A.append({
            "id_paciente": int(id_individuo),
            "tempo": float(tempo_evento),
            "pred_s": float(valor_predito),
        })

pred_event_A = pd.DataFrame(linhas_evento_A).sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_A = dir_saida / "predicao-rede-A-onehot.csv"
pred_event_A_out = pred_event_A.rename(columns={"id_paciente": "ID", "tempo": "TIME", "pred_s": "PRED_S"})
pred_event_A_out.to_csv(arquivo_saida_A, index=False)

## Rede Neural B: tempo contínuo

In [42]:
# Preparacao dos dados para a rede usando tempo continuo normalizado e categoricas one-hot

tempo_treino = pd.to_numeric(dados_treino["tempo"], errors="coerce").astype(float)

stats_tempo = {
    "mediana": float(tempo_treino.median()),
    "media": float(tempo_treino.mean()),
    "desvio": float(tempo_treino.std(ddof=0)),
}


def gera_matriz_design_tempo_continuo(dados):
    dados_ref = dados.reset_index(drop=True).copy()

    base_num = pd.DataFrame(index=dados_ref.index)

    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(
            dados_ref[col],
            estatisticas_numericas[col]
        )

    base_num["tempo"] = normaliza_coluna_numerica(
        dados_ref["tempo"],
        stats_tempo
    )

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()

        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )

        base_cat = pd.get_dummies(
            base_cat,
            prefix=covariaveis_categoricas,
            dtype=float
        ).reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    return pd.concat([base_num, base_cat], axis=1)


x_treino_B = gera_matriz_design_tempo_continuo(dados_treino)
y_treino_B = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_B = torch.tensor(x_treino_B.values.astype(np.float32))
y_train_B = torch.tensor(y_treino_B.astype(np.float32))

train_loader_B = DataLoader(
    TensorDataset(X_train_B, y_train_B),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

In [43]:
# Definição do modelo, critério e otimizador para a rede com tempos originais

model_B = rede_sobrevivencia(input_dim=X_train_B.shape[1])
criterion_B = nn.MSELoss()
optimizer_B = optim.Adam(model_B.parameters(), lr=LEARNING_RATE)

In [44]:
# Treinamento da rede B com tempos originais

for epoch_B in range(EPOCHS):
    model_B.train()
    running_loss_B = 0.0

    for x_batch_B, y_batch_B in train_loader_B:
        optimizer_B.zero_grad()
        pred_B = model_B(x_batch_B)
        loss_B = criterion_B(pred_B, y_batch_B)
        loss_B.backward()
        optimizer_B.step()
        running_loss_B += loss_B.item()

    if (epoch_B + 1) % 10 == 0:
        print(f"Época {epoch_B + 1:4d}/{EPOCHS} | mse_B={(running_loss_B / len(train_loader_B)):.6f}")

Época   10/100 | mse_B=0.044066
Época   20/100 | mse_B=0.028507
Época   30/100 | mse_B=0.022509
Época   40/100 | mse_B=0.020129
Época   50/100 | mse_B=0.018525
Época   60/100 | mse_B=0.017940
Época   70/100 | mse_B=0.017371
Época   80/100 | mse_B=0.016349
Época   90/100 | mse_B=0.015866
Época  100/100 | mse_B=0.015672


In [45]:
# Predicao da Rede B nos tempos de avaliacao do teste
x_evento_B = gera_matriz_design_tempo_continuo(dados_teste_evento)
x_evento_B = x_evento_B.reindex(columns=x_treino_B.columns, fill_value=0.0)

model_B.eval()
with torch.no_grad():
    pred_s_B = model_B(torch.tensor(x_evento_B.values.astype(np.float32))).cpu().numpy().reshape(-1)

pred_event_B = dados_teste_evento[["id_paciente", "tempo"]].copy()
pred_event_B["pred_s"] = pred_s_B
pred_event_B = pred_event_B.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_B = dir_saida / "predicao-rede-B-tempo-continuo.csv"
pred_event_B_out = pred_event_B.rename(columns={"id_paciente": "ID", "tempo": "TIME", "pred_s": "PRED_S"})
pred_event_B_out.to_csv(arquivo_saida_B, index=False)